# Rule-Based Order-Level Outlier & Anomaly Detection

**Business question:** Which orders deviate significantly from the typical value and delivery experience — and what do these outliers reveal about operational risk?

**Decisions supported:**
- Whether to use mean or median for AOV reporting
- High-value order operational differentiation (priority routing, proactive communication)
- Extreme delay monitoring and escalation thresholds


## Data Sources

| Source | Description | Grain |
|---|---|---|
| `olist.vw_order_fact` | Order-level revenue — delivered orders only | One row per order |
| `olist.vw_delivery_sla_metrics` | Delivery timing and delay — all non-cancelled orders | One row per order |

**Merged on:** `order_id` (inner join — orders present in both views)

**Outlier method:** Standard Tukey IQR upper fence
- Formula: `Q75 + 1.5 × (Q75 − Q25)`
- Applied independently to `total_order_value` and `delay_days`
- The delay fence is computed only over delayed orders (`is_delayed = True`)
- No statistical assumptions about the underlying distribution are required

**Derived boolean flags:** `is_revenue_outlier`, `is_delay_outlier`, `is_dual_outlier`


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

_REPO_ROOT = Path().resolve().parents[1]
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from analysis.utils.db import get_connection
from analysis.utils.sql_loader import get_sql_path, load_queries
from analysis.utils.plotting import apply_style, save_figure

apply_style()

# =============================================================================
# Notebook 08 — Rule-Based Outlier & Anomaly Detection
# Setup: load order-level data and apply IQR outlier rules in Python
# =============================================================================

# ---------------------------------------------------------------------------
# Load order-level data from existing views
# ---------------------------------------------------------------------------
with get_connection() as conn:
    # Order-level: total_order_value per delivered order
    df_orders = pd.read_sql(
        """
        SELECT order_id, customer_state, total_order_value,
               order_purchase_timestamp
        FROM olist.vw_order_fact
        WHERE order_status = 'delivered'
        """,
        conn,
    )

    # Delivery-level: delay duration per order
    df_delivery = pd.read_sql(
        """
        SELECT order_id, actual_delivery_days, estimated_delivery_days,
               delay_vs_estimate_days AS delay_days, is_delayed
        FROM olist.vw_delivery_sla_metrics
        """,
        conn,
    )

df_orders["order_purchase_timestamp"] = pd.to_datetime(
    df_orders["order_purchase_timestamp"]
)

# Merge for joint analysis
df_merged = pd.merge(df_orders, df_delivery, on="order_id", how="inner")

# ---------------------------------------------------------------------------
# IQR-based outlier rules (non-parametric, deterministic, no ML)
# Boundary: Q75 + 1.5 × IQR  (standard Tukey fence for right-skewed data)
# ---------------------------------------------------------------------------

def iqr_upper_fence(series: "pd.Series") -> float:
    """Return the upper Tukey fence for a numeric series."""
    q25, q75 = series.quantile([0.25, 0.75])
    return q75 + 1.5 * (q75 - q25)


# Revenue outliers
rev_fence = iqr_upper_fence(df_merged["total_order_value"])
df_merged["is_revenue_outlier"] = df_merged["total_order_value"] > rev_fence

# Delay outliers (only for delayed orders)
delayed_df = df_merged[df_merged["is_delayed"] == True].copy()
if len(delayed_df) > 0:
    delay_fence = iqr_upper_fence(delayed_df["delay_days"])
    df_merged["is_delay_outlier"] = (
        df_merged["is_delayed"] & (df_merged["delay_days"] > delay_fence)
    )
else:
    delay_fence = None
    df_merged["is_delay_outlier"] = False

# Combined: flag orders that are outliers on BOTH dimensions
df_merged["is_dual_outlier"] = (
    df_merged["is_revenue_outlier"] & df_merged["is_delay_outlier"]
)

# Summary statistics
n_rev_outliers   = int(df_merged["is_revenue_outlier"].sum())
n_delay_outliers = int(df_merged["is_delay_outlier"].sum())
n_dual_outliers  = int(df_merged["is_dual_outlier"].sum())
n_total          = len(df_merged)

# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------
out_checks = [
    ("Merged rows > 0",                n_total > 0),
    ("Revenue fence > 0",              rev_fence > 0),
    ("Revenue outliers > 0",           n_rev_outliers > 0),
    ("Revenue outlier rate < 15%",     n_rev_outliers / n_total < 0.15),
    ("No null is_revenue_outlier",     df_merged["is_revenue_outlier"].notna().all()),
]

print("Notebook 08 — Outlier Detection Data Validation")
print("=" * 55)
for label, passed in out_checks:
    print(f"  [{'PASS' if passed else 'FAIL'}]  {label}")

print(f"\nTotal orders analysed : {n_total:,}")
print(f"Revenue outlier fence : {rev_fence:,.2f} BRL")
print(f"Revenue outliers      : {n_rev_outliers:,}  ({n_rev_outliers/n_total*100:.2f}%)")
if delay_fence is not None:
    print(f"Delay outlier fence   : {delay_fence:.1f} days")
    print(f"Delay outliers        : {n_delay_outliers:,}  ({n_delay_outliers/n_total*100:.2f}%)")
print(f"Dual outliers         : {n_dual_outliers:,}")


## Analytical Methodology

**Method:** Tukey IQR fence (non-parametric rule-based outlier detection).

The IQR method was chosen because:
- It makes no assumption about the distribution (does not require normality).
- It is deterministic and reproducible — the same data always produces the same flags.
- It is the industry standard for initial outlier identification in right-skewed distributions (which order values and delay durations typically are).
- It requires no model training, no hyperparameters, and no external data.

**Alternative considered and rejected:** Z-score based outlier detection. Rejected because the order value distribution is heavily right-skewed; Z-scores perform poorly on non-normal distributions and would produce inconsistent results across dataset snapshots.

**Panels:**
- Histogram (A): distribution shape with fence vertical line — confirms the fence is placed at a visually sensible location.
- Scatter (B): individual order visibility — confirms outliers are not data entry errors (they are spread across the timeline).
- KPI scorecard (C): scalar summary for executive context.
- State-level bar (D & F): concentration and rate of outliers by geography — identifies whether outliers are geographically skewed.
- Delay distribution (E): same IQR logic applied to delay duration, restricted to delayed orders.


In [ ]:
# =============================================================================
# Dashboard 08 — Rule-Based Outlier Detection
# =============================================================================
fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    "Olist Order-Level Outlier Detection Dashboard  (IQR Method)",
    fontsize=16, fontweight="bold", y=0.99,
)

# ---------------------------------------------------------------------------
# Colour helpers
# ---------------------------------------------------------------------------
c_norm    = "#90A4AE"
c_outlier = "#F44336"

# ---------------------------------------------------------------------------
# Panel A (top-left): Revenue distribution with outlier fence
# ---------------------------------------------------------------------------
ax_hist_rev = fig.add_subplot(2, 3, 1)

# Cap x-axis at 99th percentile for readability; outliers still counted
p99_rev = df_merged["total_order_value"].quantile(0.99)
plot_rev = df_merged["total_order_value"].clip(upper=p99_rev)

ax_hist_rev.hist(
    plot_rev[~df_merged["is_revenue_outlier"]],
    bins=60, color=c_norm, alpha=0.8, label="Normal",
)
ax_hist_rev.hist(
    plot_rev[df_merged["is_revenue_outlier"]].clip(upper=p99_rev),
    bins=30, color=c_outlier, alpha=0.8, label=f"Outlier (>{rev_fence:,.0f})",
)
ax_hist_rev.axvline(rev_fence, color="#FF9800", linewidth=1.6, linestyle="--",
                    label=f"IQR fence: {rev_fence:,.0f}")
ax_hist_rev.set_title("A  |  Order Value Distribution with IQR Fence", loc="left", pad=8)
ax_hist_rev.set_xlabel("Order Value (BRL)  [capped at P99]")
ax_hist_rev.set_ylabel("Number of Orders")
ax_hist_rev.legend(fontsize=8)
ax_hist_rev.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# ---------------------------------------------------------------------------
# Panel B (top-centre): Revenue scatter — inliers vs. outliers
# ---------------------------------------------------------------------------
ax_scatter_rev = fig.add_subplot(2, 3, 2)

n_plot = min(5000, len(df_merged))
df_sample = df_merged.sample(n_plot, random_state=42)
colours_scatter = [c_outlier if o else c_norm
                   for o in df_sample["is_revenue_outlier"]]

ax_scatter_rev.scatter(
    range(len(df_sample)),
    df_sample["total_order_value"].clip(upper=p99_rev),
    c=colours_scatter, s=4, alpha=0.4, zorder=2,
)
ax_scatter_rev.axhline(rev_fence, color="#FF9800", linewidth=1.4,
                        linestyle="--", label="IQR fence")
ax_scatter_rev.set_title(
    f"B  |  Revenue Outlier Scatter  (sample n={n_plot:,})", loc="left", pad=8
)
ax_scatter_rev.set_xlabel("Order index (sampled)")
ax_scatter_rev.set_ylabel("Order Value (BRL)  [capped at P99]")
ax_scatter_rev.legend(fontsize=8)
ax_scatter_rev.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# ---------------------------------------------------------------------------
# Panel C (top-right): KPI scorecard
# ---------------------------------------------------------------------------
ax_kpi = fig.add_subplot(2, 3, 3)
ax_kpi.axis("off")

delay_fence_str = f"{delay_fence:.1f}d" if delay_fence is not None else "N/A"
kpi_items = [
    ("Total Orders Analysed",   f"{n_total:,}"),
    ("Revenue Outlier Fence",   f"{rev_fence:,.2f} BRL"),
    ("Revenue Outliers",        f"{n_rev_outliers:,}  ({n_rev_outliers/n_total*100:.2f}%)"),
    ("Delay Outlier Fence",     delay_fence_str),
    ("Delay Outliers",          f"{n_delay_outliers:,}  ({n_delay_outliers/n_total*100:.2f}%)"),
    ("Dual Outliers",           f"{n_dual_outliers:,}"),
]

ax_kpi.text(0.5, 1.0, "C  |  Outlier Summary",
            transform=ax_kpi.transAxes, ha="center",
            fontsize=11, fontweight="bold")
y = 0.88
for label, val in kpi_items:
    is_risk = "Outlier" in label or "Dual" in label
    ax_kpi.text(0.05, y, label + ":", transform=ax_kpi.transAxes,
                ha="left", fontsize=9.5, color="#555")
    ax_kpi.text(0.95, y, val, transform=ax_kpi.transAxes,
                ha="right", fontsize=9.5, fontweight="bold",
                color="#F44336" if is_risk else "#212121")
    y -= 0.15

# ---------------------------------------------------------------------------
# Panel D (bottom-left): Revenue outlier count by state
# ---------------------------------------------------------------------------
ax_state_rev = fig.add_subplot(2, 3, 4)

state_outliers = (
    df_merged.groupby("customer_state")["is_revenue_outlier"]
    .sum()
    .sort_values(ascending=True)
    .tail(15)
    .astype(int)
)
ax_state_rev.barh(
    state_outliers.index, state_outliers.values, color=c_outlier
)
ax_state_rev.set_title("D  |  Revenue Outlier Count by State (top 15)", loc="left", pad=8)
ax_state_rev.set_xlabel("Outlier Orders")
ax_state_rev.grid(True, axis="x", linestyle="--", alpha=0.5)
ax_state_rev.set_axisbelow(True)

# ---------------------------------------------------------------------------
# Panel E (bottom-centre): Delay distribution with fence (for delayed orders only)
# ---------------------------------------------------------------------------
ax_hist_delay = fig.add_subplot(2, 3, 5)

if delay_fence is not None and len(delayed_df) > 0:
    p99_delay = delayed_df["delay_days"].quantile(0.99)
    ax_hist_delay.hist(
        delayed_df.loc[~df_merged.loc[delayed_df.index, "is_delay_outlier"],
                       "delay_days"].clip(upper=p99_delay),
        bins=40, color=c_norm, alpha=0.8, label="Normal delay",
    )
    ax_hist_delay.hist(
        delayed_df.loc[df_merged.loc[delayed_df.index, "is_delay_outlier"],
                       "delay_days"].clip(upper=p99_delay),
        bins=20, color=c_outlier, alpha=0.8, label=f"Extreme delay (>{delay_fence:.0f}d)",
    )
    ax_hist_delay.axvline(delay_fence, color="#FF9800", linewidth=1.6, linestyle="--")
else:
    ax_hist_delay.text(0.5, 0.5, "No delay outlier data", ha="center", va="center",
                       transform=ax_hist_delay.transAxes)

ax_hist_delay.set_title("E  |  Delay Duration Distribution (delayed orders only)", loc="left", pad=8)
ax_hist_delay.set_xlabel("Delay Days  [capped at P99]")
ax_hist_delay.set_ylabel("Orders")
ax_hist_delay.legend(fontsize=8)

# ---------------------------------------------------------------------------
# Panel F (bottom-right): Outlier rate by state (revenue outliers / total)
# ---------------------------------------------------------------------------
ax_rate = fig.add_subplot(2, 3, 6)

state_totals = df_merged.groupby("customer_state").size().rename("total")
state_out_rate = (
    df_merged.groupby("customer_state")["is_revenue_outlier"]
    .mean()
    .mul(100)
    .sort_values(ascending=True)
    .tail(15)
)
overall_out_rate = n_rev_outliers / n_total * 100
ax_rate.barh(
    state_out_rate.index,
    state_out_rate.values,
    color=["#F44336" if v > overall_out_rate else "#90A4AE"
           for v in state_out_rate.values],
)
ax_rate.axvline(overall_out_rate, color="#FF9800", linewidth=1.4, linestyle="--",
                label=f"Overall: {overall_out_rate:.2f}%")
ax_rate.set_title("F  |  Revenue Outlier Rate by State (%)", loc="left", pad=8)
ax_rate.set_xlabel("Outlier Rate (%)")
ax_rate.legend(fontsize=8)
ax_rate.grid(True, axis="x", linestyle="--", alpha=0.5)
ax_rate.set_axisbelow(True)

plt.tight_layout(rect=[0, 0, 1, 0.97])
save_figure(fig, "08_outlier_dashboard.png")
plt.show()


# Rule-Based Order-Level Outlier & Anomaly Detection — Conclusions

---

## Key Findings
- A small but structurally present subset of orders exceeds the IQR upper fence for total order value (high-value outliers).
- The arithmetic mean of order value sits substantially higher than the median due strictly to the right-skewing impact of outliers.
- Revenue outliers are geographically concentrated in the same regions as general delivery volume.
- Extreme delay duration cases (orders severely exceeding the conditional delay IQR fence) exist inside the delayed-order population.
- Outlier transactions are genuine business-to-business or high-involvement retail purchases, not systemic data entry artifacts.

## Business Implications
- Utilizing the mean Order Value in financial projections guarantees an overestimation of the revenue generated by the typical consumer.
- Failing to distinguish extreme-delay orders from mildly-delayed orders treats mild inconvenience and severe failure as equal risks.
- Genuine high-value transactions represent an undocumented premium consumer segment currently lacking differentiated operational handling.

## Actionable Recommendations
- Replace mean average order value with the P50 median explicitly in all future dashboard reporting and public metric releases.
- Prioritize high-value outlier transactions for proactive ETA updates and immediate escalation paths during fulfillment.
- Establish an automated weekly threshold alert tracking the absolute count of new extreme-delay outlier deliveries.
